# FPR Annotation: Inter-Annotator Agreement and False Positive Rate Analysis

This notebook processes two completed FPR annotation files and the join table to compute:
- Inter-annotator agreement (Cohen's κ, percent agreement)
- Per-level FPR estimates with 95% Wilson score confidence intervals
- Comparison of annotator judgments against benchmark labels
- Publication-ready metrics and diagnostic visualizations

## Cell 0: Imports and Configuration

All magic numbers and path constants are defined here. Edit this cell to update paths or thresholds.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from sklearn.metrics import cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay
from statsmodels.stats.proportion import proportion_confint
from pathlib import Path

# --- Paths (relative to this notebook's directory: fpr_annotation/) ---
ANNOTATOR1_PATH = Path("completed_labels/annotator1.tsv")
ANNOTATOR2_PATH = Path("completed_labels/annotator2.tsv")
JOIN_TABLE_PATH = Path("fpr_join_table.csv")
FIG1_OUT_PATH = Path("outputs/fig14_fpr_by_level.pdf")
FIG2_OUT_PATH = Path("outputs/fig15_fpr_agreement_matrix.pdf")
FIG3_OUT_PATH = Path("outputs/fig_unused_fpr_unclear_rate.pdf")
FIG4_OUT_PATH = Path("outputs/fig16_fpr_annotator_vs_benchmark.pdf")
METRICS_OUT_PATH = Path("outputs/fpr_reportable_metrics.txt")

# --- Judgment label constants ---
HATEFUL     = "Hateful"
NOT_HATEFUL = "Not Hateful"
UNCLEAR     = "Unclear"
VALID_JUDGMENTS = {HATEFUL, NOT_HATEFUL, UNCLEAR}

# Synonyms to normalize during loading (case-insensitive, whitespace-stripped)
# Annotators wrote "Unknown" instead of "Unclear" — map it consistently.
JUDGMENT_ALIASES = {
    "unknown": UNCLEAR,
    "hateful": HATEFUL,
    "not hateful": NOT_HATEFUL,
    "unclear": UNCLEAR,
}

# --- Coding levels ---
LEVELS = ["L2", "L3", "L4"]

# --- Stability threshold: report warning if fewer than this many agreed items at a level ---
MIN_AGREED_ITEMS = 20

# --- Confidence interval parameters ---
CI_METHOD = "wilson"  # Wilson score interval is preferred for proportions near 0 or 1
CI_ALPHA  = 0.05      # 95% CI

# --- Figure style ---
# Okabe-Ito colorblind-safe palette (8 colors)
CB_COLORS = [
    "#E69F00", "#56B4E9", "#009E73",
    "#F0E442", "#0072B2", "#D55E00", "#CC79A7", "#000000"
]

print("Configuration loaded.")


## Cell 1: Load and Validate Data

Load the three input files, normalize judgment labels, and run structural validation checks.
Annotator 2's TSV has duplicate column headers from Excel export; we pick the correct one.
Both annotators wrote "Unknown" instead of "Unclear" — normalize via `JUDGMENT_ALIASES`.

In [ ]:
def normalize_judgment(raw):
    """
    Normalize a raw judgment string to a canonical label.

    Strips surrounding whitespace, lowercases, and maps via JUDGMENT_ALIASES.
    Returns the canonical label string, or pd.NA if input is missing/unrecognized.

    Args:
        raw: raw judgment value (str, float NaN, or pd.NA)

    Returns:
        Canonical judgment string or pd.NA
    """
    if pd.isna(raw):
        return pd.NA
    normalized = str(raw).strip().lower()
    return JUDGMENT_ALIASES.get(normalized, pd.NA)


def load_annotator1(path):
    """
    Load annotator 1's TSV. Column layout is standard: row_id, ..., judgment, annotator_notes.

    Args:
        path: Path to the TSV file

    Returns:
        DataFrame with columns [row_id, judgment, annotator_notes]
    """
    df = pd.read_csv(path, sep="\t", dtype={"row_id": int})
    df["judgment"] = df["judgment"].apply(normalize_judgment)
    return df[["row_id", "judgment", "annotator_notes"]]


def load_annotator2(path):
    """
    Load annotator 2's TSV. This file has duplicate column headers from an Excel round-trip;
    the actual judgment lives in 'judgement.1' (the second occurrence). We rename it to
    'judgment' for consistency with annotator 1.

    Args:
        path: Path to the TSV file

    Returns:
        DataFrame with columns [row_id, judgment, annotator_notes]
    """
    df = pd.read_csv(path, sep="\t", dtype={"row_id": int})
    # 'judgement.1' is the populated column; 'judgement' is empty (artifact of duplicate headers)
    if "judgement.1" in df.columns:
        df = df.rename(columns={"judgement.1": "judgment", "annotator_notes.1": "annotator_notes_a2"})
    elif "judgement" in df.columns:
        df = df.rename(columns={"judgement": "judgment"})
    df["judgment"] = df["judgment"].apply(normalize_judgment)
    notes_col = "annotator_notes_a2" if "annotator_notes_a2" in df.columns else "annotator_notes"
    return df[["row_id", "judgment", notes_col]].rename(columns={notes_col: "annotator_notes"})


# Load all three files
a1 = load_annotator1(ANNOTATOR1_PATH)
a2 = load_annotator2(ANNOTATOR2_PATH)
jt = pd.read_csv(JOIN_TABLE_PATH, dtype={"row_id": int, "binary_hate": int})

# --- Validation ---

# 1. No duplicate row_ids within each file
assert a1["row_id"].is_unique, "Annotator 1 has duplicate row_ids"
assert a2["row_id"].is_unique, "Annotator 2 has duplicate row_ids"
assert jt["row_id"].is_unique, "Join table has duplicate row_ids"

# 2. Both annotation files share the same row_ids
assert set(a1["row_id"]) == set(a2["row_id"]), "Annotator row_id sets differ"

# 3. Join table row_ids match annotation row_ids
assert set(jt["row_id"]) == set(a1["row_id"]), "Join table row_ids do not match annotation row_ids"

# 4. Check for invalid or missing judgments (warn, don't error)
invalid_a1 = a1[a1["judgment"].isna()]
invalid_a2 = a2[a2["judgment"].isna()]
if len(invalid_a1):
    print(f"WARNING: {len(invalid_a1)} invalid/missing judgments in Annotator 1 (row_ids: {invalid_a1['row_id'].tolist()})")
if len(invalid_a2):
    print(f"WARNING: {len(invalid_a2)} invalid/missing judgments in Annotator 2 (row_ids: {invalid_a2['row_id'].tolist()})")

# --- Data summary ---
print(f"Annotator 1: {len(a1)} rows loaded, {a1['judgment'].isna().sum()} missing judgments")
print(f"Annotator 2: {len(a2)} rows loaded, {a2['judgment'].isna().sum()} missing judgments")
print(f"Join table:  {len(jt)} rows loaded")
print()
print("Coding level distribution in join table:")
level_counts = jt["coding_level"].value_counts().sort_index()
print("  " + "  |  ".join(f"{lvl}: {cnt}" for lvl, cnt in level_counts.items()))

## Cell 2: Merge and Classify Agreement

Merge both annotation files and the join table into a single working dataframe.
Then classify each row by agreement category. The categories distinguish hard
disagreement (Hateful vs Not Hateful) from soft ambiguity (one or both Unclear),
which matters for how we handle them in FPR computation.

In [ ]:
def classify_agreement(j1, j2):
    """
    Classify a pair of judgments into an agreement category.

    Categories:
      agree_hateful      — both said Hateful
      agree_not_hateful  — both said Not Hateful
      both_unclear       — both said Unclear (soft: no definitive signal from either)
      one_unclear        — one definitive, one Unclear (partial signal)
      disagree           — one Hateful, one Not Hateful (hard disagreement, no Unclear)

    Args:
        j1: judgment from annotator 1 (str)
        j2: judgment from annotator 2 (str)

    Returns:
        agreement category string
    """
    if j1 == j2:
        if j1 == UNCLEAR:
            return "both_unclear"
        return "agree_hateful" if j1 == HATEFUL else "agree_not_hateful"
    if UNCLEAR in (j1, j2):
        return "one_unclear"
    return "disagree"  # one Hateful, one Not Hateful — substantive disagreement


# Merge annotation files on row_id
df = (
    a1.rename(columns={"judgment": "judgment_a1", "annotator_notes": "notes_a1"})
    .merge(
        a2.rename(columns={"judgment": "judgment_a2", "annotator_notes": "notes_a2"}),
        on="row_id",
        how="inner",
    )
    .merge(
        jt[["row_id", "coding_level", "all_target_groups", "binary_hate", "matched_surface_form", "dogwhistle", "dataset"]],
        on="row_id",
        how="inner",
    )
)

# Verify no rows were lost in the merge
assert len(df) == len(a1), f"Row count changed after merge: {len(df)} vs {len(a1)}"

# Classify agreement
df["agreement_class"] = df.apply(
    lambda row: classify_agreement(row["judgment_a1"], row["judgment_a2"]), axis=1
)

print(f"Working dataframe: {len(df)} rows, columns: {list(df.columns)}")
print()
print("Agreement class distribution:")
print(df["agreement_class"].value_counts().to_string())
print()
print("Crosstab: judgment_a1 × judgment_a2")
crosstab = pd.crosstab(
    df["judgment_a1"], df["judgment_a2"],
    rownames=["Annotator 1"], colnames=["Annotator 2"],
    margins=True,
)
print(crosstab)

## Cell 3: Overall Inter-Annotator Agreement Metrics

We report five complementary IAA metrics:
- **Percent agreement**: simple, interpretable, but inflated by chance.
- **Cohen's κ (3-way)**: chance-corrected agreement on all three labels; primary reportable metric.
- **Cohen's κ (binary, excl. Unclear)**: agreement only on definitive judgments; isolates annotator
  reliability on the items they felt confident about.
- **Per-level κ**: whether agreement varies by coding sophistication (L2/L3/L4).
- **Unclear rates**: measures annotator uncertainty, which affects effective sample size.

In [ ]:
def compute_kappa_safe(labels_a, labels_b, label_desc=""):
    """
    Compute Cohen's κ with error handling for degenerate cases.

    sklearn's cohen_kappa_score raises ValueError if one annotator uses only one
    label; we catch this and return NaN with a warning.

    Args:
        labels_a: array-like of labels from annotator 1
        labels_b: array-like of labels from annotator 2
        label_desc: description string for warning messages

    Returns:
        float kappa value, or np.nan if computation fails
    """
    try:
        return cohen_kappa_score(labels_a, labels_b)
    except ValueError as e:
        print(f"WARNING: kappa computation failed ({label_desc}): {e}")
        return np.nan


# A. Percent agreement (includes both_unclear as agreement)
pct_agree = (df["judgment_a1"] == df["judgment_a2"]).mean()

# B. Cohen's κ — three-way (Hateful / Not Hateful / Unclear), all rows
kappa_3way = compute_kappa_safe(df["judgment_a1"], df["judgment_a2"], "3-way overall")

# C. Cohen's κ — binary, only rows where NEITHER annotator said Unclear
#    Recode: Hateful=1, Not Hateful=0 for binary κ
mask_definitive = (df["judgment_a1"] != UNCLEAR) & (df["judgment_a2"] != UNCLEAR)
df_def = df[mask_definitive].copy()
recode = {HATEFUL: 1, NOT_HATEFUL: 0}
kappa_binary = compute_kappa_safe(
    df_def["judgment_a1"].map(recode),
    df_def["judgment_a2"].map(recode),
    "binary excl. Unclear",
)
n_definitive = len(df_def)

# D. Per-level Cohen's κ (three-way)
level_kappas = {}
level_n = {}
for lvl in LEVELS:
    sub = df[df["coding_level"] == lvl]
    level_n[lvl] = len(sub)
    level_kappas[lvl] = compute_kappa_safe(sub["judgment_a1"], sub["judgment_a2"], f"3-way {lvl}")

# E. Unclear rates — per annotator, overall and per level
def unclear_rate(series):
    """Proportion of judgments that are Unclear."""
    return (series == UNCLEAR).mean()

unclear_a1_overall = unclear_rate(df["judgment_a1"])
unclear_a2_overall = unclear_rate(df["judgment_a2"])
both_unclear_overall = (df["agreement_class"] == "both_unclear").mean()
one_unclear_overall  = (df["agreement_class"] == "one_unclear").mean()

unclear_by_level = {}
for lvl in LEVELS:
    sub = df[df["coding_level"] == lvl]
    unclear_by_level[lvl] = {
        "a1":   unclear_rate(sub["judgment_a1"]),
        "a2":   unclear_rate(sub["judgment_a2"]),
        "both": (sub["agreement_class"] == "both_unclear").mean(),
        "one":  (sub["agreement_class"] == "one_unclear").mean(),
    }

# --- Print summary ---
print("=" * 60)
print("=== OVERALL IAA ===")
print(f"Total items:                    {len(df)}")
print(f"Percent agreement:              {pct_agree*100:.1f}%  (includes both-Unclear)")
print(f"Cohen's κ (3-way):              {kappa_3way:.3f}")
print(f"Cohen's κ (binary, excl. Unclear): {kappa_binary:.3f}  (N={n_definitive} items)")
print()
print("=== PER-LEVEL κ ===")
print(f"{'Level':<7} {'N':<6} {'κ (3-way)':<12} {'Status'}")
print("-" * 40)
for lvl in LEVELS:
    status = "stable" if level_n[lvl] >= MIN_AGREED_ITEMS else "UNSTABLE"
    k = level_kappas[lvl]
    k_str = f"{k:.3f}" if not np.isnan(k) else "N/A"
    print(f"{lvl:<7} {level_n[lvl]:<6} {k_str:<12} [{status}]")
print()
print("=== UNCLEAR RATES ===")
header = f"{'':12} {'Overall':>8}" + "".join(f"  {lvl:>6}" for lvl in LEVELS)
print(header)
rows_data = [
    ("Ann. 1:",  unclear_a1_overall,  {lvl: unclear_by_level[lvl]["a1"]  for lvl in LEVELS}),
    ("Ann. 2:",  unclear_a2_overall,  {lvl: unclear_by_level[lvl]["a2"]  for lvl in LEVELS}),
    ("Both:",    both_unclear_overall, {lvl: unclear_by_level[lvl]["both"] for lvl in LEVELS}),
    ("One:",     one_unclear_overall,  {lvl: unclear_by_level[lvl]["one"]  for lvl in LEVELS}),
]
for label, overall, by_lvl in rows_data:
    row_str = f"{label:<12} {overall*100:>7.1f}%"
    row_str += "".join(f"  {by_lvl[lvl]*100:>5.1f}%" for lvl in LEVELS)
    print(row_str)

## Cell 4: Resolved Labels and FPR Computation

### Step 1: Resolve disagreements
We apply a principled resolution scheme to handle the three classes of non-agreement:
- **One Unclear, one definitive**: use the definitive label. Rationale: one annotator had enough
  context to make a judgment; treating uncertainty as abstention would discard real signal.
- **Hard disagreement (Hateful vs Not Hateful)**: exclude. No principled resolution without
  adjudication; including either would inject noise.
- **Both Unclear**: exclude. Neither annotator could decide; no denominator contribution.

### Step 2: FPR with Wilson CIs
FPR = proportion of *included* items judged Not Hateful. We use Wilson score CIs because
they have better coverage than Wald CIs when FPR is near 0 or 1, which is likely for
highly specific coding levels.

In [ ]:
def resolve_judgment(row):
    """
    Apply the resolution scheme to produce a single label from two annotator judgments.

    Resolution rules (in priority order):
      1. Both agree on a definitive label → use that label
      2. One Unclear, one definitive → use the definitive label
      3. Both Unclear → exclude (flag excluded_both_unclear)
      4. Hard disagreement → exclude (flag excluded_disagreement)

    Args:
        row: DataFrame row with judgment_a1, judgment_a2, agreement_class

    Returns:
        tuple (resolved_judgment, resolution_method)
    """
    j1, j2, ac = row["judgment_a1"], row["judgment_a2"], row["agreement_class"]

    if ac == "agree_hateful":
        return HATEFUL, "both_agree_hateful"
    if ac == "agree_not_hateful":
        return NOT_HATEFUL, "both_agree_not_hateful"
    if ac == "both_unclear":
        return UNCLEAR, "excluded_both_unclear"
    if ac == "one_unclear":
        # Use whichever annotator gave a definitive label
        definitive = j1 if j1 != UNCLEAR else j2
        return definitive, "one_unclear_→_definitive"
    # ac == "disagree": hard disagreement
    return pd.NA, "excluded_disagreement"


resolved_cols = df.apply(resolve_judgment, axis=1, result_type="expand")
df["resolved_judgment"] = resolved_cols[0]
df["resolution_method"] = resolved_cols[1]

# --- Resolution summary ---
method_counts = df["resolution_method"].value_counts()
total = len(df)
n_included = df["resolution_method"].isin(["both_agree_hateful", "both_agree_not_hateful", "one_unclear_→_definitive"]).sum()
n_excluded = total - n_included

print(f"{'Resolution method':<30} {'N':>5}  {'%':>6}")
print("-" * 45)
display_order = [
    "both_agree_hateful", "both_agree_not_hateful",
    "one_unclear_→_definitive", "excluded_disagreement", "excluded_both_unclear"
]
for method in display_order:
    n = method_counts.get(method, 0)
    note = "  (included in FPR)" if "definitive" in method else ""
    note = "  (excluded from FPR)" if "excluded" in method else note
    print(f"{method:<30} {n:>5}  {n/total*100:>5.1f}%{note}")
print("-" * 45)
print(f"{'Included in FPR:':<30} {n_included:>5}  {n_included/total*100:>5.1f}%")
print(f"{'Excluded from FPR:':<30} {n_excluded:>5}  {n_excluded/total*100:>5.1f}%")

# --- Step 2: Per-level FPR with Wilson CIs ---

# Only included rows contribute to the FPR denominator
df_included = df[df["resolved_judgment"].isin([HATEFUL, NOT_HATEFUL])].copy()

fpr_results = []
for lvl in LEVELS:
    sub_all   = df[df["coding_level"] == lvl]
    sub_inc   = df_included[df_included["coding_level"] == lvl]
    n_inc     = len(sub_inc)
    n_exc     = len(sub_all) - n_inc
    n_not_hat = (sub_inc["resolved_judgment"] == NOT_HATEFUL).sum()
    n_both_unc = (sub_all["resolution_method"] == "excluded_both_unclear").sum()

    if n_inc == 0:
        fpr_results.append({"level": lvl, "n_included": 0, "n_excluded": n_exc,
                             "fpr": np.nan, "ci_lo": np.nan, "ci_hi": np.nan,
                             "unclear_pct": np.nan})
        print(f"WARNING: No included rows at level {lvl} — FPR cannot be computed")
        continue

    if n_inc < MIN_AGREED_ITEMS:
        print(f"WARNING: Only {n_inc} included items at {lvl} — FPR estimate is unstable")

    fpr = n_not_hat / n_inc
    ci_lo, ci_hi = proportion_confint(count=n_not_hat, nobs=n_inc, alpha=CI_ALPHA, method=CI_METHOD)
    unclear_pct = n_both_unc / len(sub_all)

    fpr_results.append({
        "level": lvl, "n_included": n_inc, "n_excluded": n_exc,
        "fpr": fpr, "ci_lo": ci_lo, "ci_hi": ci_hi,
        "unclear_pct": unclear_pct,
    })

fpr_df = pd.DataFrame(fpr_results)

print()
print("=== FPR ESTIMATES (95% Wilson CI) ===")
print(f"{'Level':<7} {'N_inc':>7} {'N_exc':>7} {'FPR':>7} {'CI_lo':>7} {'CI_hi':>7} {'Unclear%':>9}")
print("-" * 55)
for _, row in fpr_df.iterrows():
    status = " [UNSTABLE]" if row["n_included"] < MIN_AGREED_ITEMS else ""
    print(
        f"{row['level']:<7} {row['n_included']:>7} {row['n_excluded']:>7} "
        f"{row['fpr']:>7.3f} {row['ci_lo']:>7.3f} {row['ci_hi']:>7.3f} "
        f"{row['unclear_pct']*100:>8.1f}%{status}"
    )

# Monotonic test: check if FPR increases from L2 → L3 → L4
fpr_vals = fpr_df.set_index("level")["fpr"]
mono = all(fpr_vals[LEVELS[i]] <= fpr_vals[LEVELS[i+1]] for i in range(len(LEVELS)-1))
strict_mono = all(fpr_vals[LEVELS[i]] < fpr_vals[LEVELS[i+1]] for i in range(len(LEVELS)-1))
if strict_mono:
    mono_label = "YES"
elif mono:
    mono_label = "PARTIAL (non-strict)"
else:
    mono_label = "NO"
print()
ordering = " < ".join(f"{lvl}({fpr_vals[lvl]:.3f})" for lvl in LEVELS)
print(f"Monotonic test: L2 FPR < L3 FPR < L4 FPR? [{mono_label}]")
print(f"  Actual ordering: {ordering}")

## Cell 5: Benchmark Label Comparison

Compare human-resolved judgments against the `binary_hate` labels from the original
benchmark datasets. This validates the key claim of the paper: benchmark labels
systematically mislabel posts containing dogwhistles.

- **FP (benchmark perspective)**: annotators say Hateful, benchmark says 0 (non-hateful)
  → the benchmark missed a hateful post
- **FN (benchmark perspective)**: annotators say Not Hateful, benchmark says 1 (hateful)
  → the benchmark over-labeled a non-hateful post

The direction of disagreement tells us whether benchmark failures align with our
Case A (over-labeling) or Case B (under-labeling) hypotheses.

In [ ]:
# Use only included rows (resolved_judgment is Hateful or Not Hateful)
df_comp = df_included.copy()

# Recode resolved judgment to binary: Hateful=1, Not Hateful=0
df_comp["resolved_binary"] = (df_comp["resolved_judgment"] == HATEFUL).astype(int)

# A. Overall agreement rate between resolved judgment and binary_hate
overall_agree_rate = (df_comp["resolved_binary"] == df_comp["binary_hate"]).mean()

# B. Per-level agreement rate
level_agree = {}
level_agree_n = {}
for lvl in LEVELS:
    sub = df_comp[df_comp["coding_level"] == lvl]
    level_agree[lvl] = (sub["resolved_binary"] == sub["binary_hate"]).mean()
    level_agree_n[lvl] = len(sub)

# C. Confusion matrix
# Rows = annotator resolved, Cols = benchmark
# TP: both 1 (hateful), FP: ann=1 bench=0, FN: ann=0 bench=1, TN: both 0
tp = ((df_comp["resolved_binary"] == 1) & (df_comp["binary_hate"] == 1)).sum()
fp = ((df_comp["resolved_binary"] == 1) & (df_comp["binary_hate"] == 0)).sum()
fn = ((df_comp["resolved_binary"] == 0) & (df_comp["binary_hate"] == 1)).sum()
tn = ((df_comp["resolved_binary"] == 0) & (df_comp["binary_hate"] == 0)).sum()

# Store confusion matrix for visualization in Cell 6
cm_array = np.array([[tp, fp], [fn, tn]])

print("=== ANNOTATOR vs BENCHMARK LABEL AGREEMENT ===")
print(f"Overall agreement (resolved vs binary_hate): {overall_agree_rate*100:.1f}%  (N={len(df_comp)})")
print()
print("Per level:")
for lvl in LEVELS:
    print(f"  {lvl}: {level_agree[lvl]*100:.1f}%  (N={level_agree_n[lvl]})")
print()
print("Confusion matrix (resolved annotation vs benchmark):")
print(f"{'':30} {'Benchmark: Hateful':>20} {'Benchmark: Not Hateful':>23}")
print(f"{'Ann: Hateful':<30} {tp:>20}  (TP)    {fp:>14}  (FP)")
print(f"{'Ann: Not Hateful':<30} {fn:>20}  (FN)    {tn:>14}  (TN)")
print()
print("Interpretation:")
print("  FP cases: benchmark labeled non-hateful, but annotators judged Hateful")
print("            → benchmark annotation FAILURE (Case A: false negatives in the benchmark)")
print("  FN cases: benchmark labeled hateful, but annotators judged Not Hateful")
print("            → benchmark over-labeling (Case B: false positives in the benchmark)")
print(f"  Total FPR (annotator perspective): {(fp+tn)/(tp+fp+fn+tn):.3f}  ")
print(f"  (proportion benchmark called non-hateful across all included items)")

## Cell 6: Visualizations

A four-panel publication-ready figure. All panels use colorblind-safe palettes.
- **Panel 1**: Per-level FPR with Wilson CI error bars — the key quantitative result.
- **Panel 2**: Annotator agreement matrix — IAA diagnostic.
- **Panel 3**: Per-level Unclear rates by annotator — uncertainty diagnostic.
- **Panel 4**: Stacked bar of resolved vs benchmark label agreement — validation plot.

In [ ]:
# ── ACL/EMNLP figure style ─────────────────────────────────────────────────
# Matches the conventions used elsewhere in this repo (generate_figures_final.py):
# no titles (captions carry that role in ACL format), 10pt axis labels,
# 8pt tick/legend/annotation text, no top/right spines.
ACL_AXIS_LABEL_FONTSIZE = 10
ACL_TICK_LABEL_FONTSIZE = 8
ACL_LEGEND_FONTSIZE = 8
ACL_ANNOTATION_FONTSIZE = 8

plt.rcParams.update({
    "font.size": ACL_TICK_LABEL_FONTSIZE,
    "axes.labelsize": ACL_AXIS_LABEL_FONTSIZE,
    "xtick.labelsize": ACL_TICK_LABEL_FONTSIZE,
    "ytick.labelsize": ACL_TICK_LABEL_FONTSIZE,
    "legend.fontsize": ACL_LEGEND_FONTSIZE,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 300,
})

LEVEL_COLORS = [CB_COLORS[0], CB_COLORS[1], CB_COLORS[2]]  # amber, sky-blue, teal

# ── Figure 1: Per-level FPR with 95% CI error bars ────────────────────────────
fig1, ax1 = plt.subplots(figsize=(7, 5.5))

fprs   = [fpr_df.set_index("level").loc[lvl, "fpr"]   for lvl in LEVELS]
ci_los = [fpr_df.set_index("level").loc[lvl, "ci_lo"] for lvl in LEVELS]
ci_his = [fpr_df.set_index("level").loc[lvl, "ci_hi"] for lvl in LEVELS]
err_lo = [f - lo for f, lo in zip(fprs, ci_los)]
err_hi = [hi - f  for f, hi in zip(fprs, ci_his)]
n_incs = [fpr_df.set_index("level").loc[lvl, "n_included"] for lvl in LEVELS]

bars = ax1.bar(LEVELS, fprs, color=LEVEL_COLORS, width=0.5, zorder=2)
ax1.errorbar(
    LEVELS, fprs,
    yerr=[err_lo, err_hi],
    fmt="none", color="black", capsize=6, linewidth=1.5, zorder=3,
)
ax1.axhline(0, color="gray", linestyle="--", linewidth=0.8, zorder=1)
ax1.axhline(1, color="gray", linestyle="--", linewidth=0.8, zorder=1)
ax1.set_ylim(-0.05, 1.10)
ax1.set_ylabel("False Positive Rate", fontsize=ACL_AXIS_LABEL_FONTSIZE)
ax1.set_title("")

for bar, fpr_val, n_inc in zip(bars, fprs, n_incs):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        fpr_val + max(err_hi) + 0.04,
        f"{fpr_val:.3f}\n(N={int(n_inc)})",
        ha="center", va="bottom", fontsize=ACL_ANNOTATION_FONTSIZE,
    )

fig1.tight_layout(pad=2.0)
fig1.savefig(FIG1_OUT_PATH, format="pdf", bbox_inches="tight")
plt.show()
print(f"Figure 1 saved to {FIG1_OUT_PATH}")

# ── Figure 2: Annotator agreement heatmap ─────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(7, 5.5))

cat_order = [HATEFUL, NOT_HATEFUL, UNCLEAR]
heatmap_data = pd.crosstab(
    pd.Categorical(df["judgment_a1"], categories=cat_order, ordered=True),
    pd.Categorical(df["judgment_a2"], categories=cat_order, ordered=True),
).rename_axis(index="Annotator 1", columns="Annotator 2")

heatmap_pct = heatmap_data / heatmap_data.values.sum()
annot_labels = np.array([
    [f"{heatmap_data.iloc[i,j]}\n({heatmap_pct.iloc[i,j]*100:.1f}%)"
     for j in range(heatmap_data.shape[1])]
    for i in range(heatmap_data.shape[0])
])

sns.heatmap(
    heatmap_data, ax=ax2,
    annot=annot_labels, fmt="", cmap="Blues",
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Count"},
    xticklabels=cat_order, yticklabels=cat_order,
    annot_kws={"fontsize": ACL_ANNOTATION_FONTSIZE},
)
ax2.set_title("")
ax2.tick_params(axis="x", rotation=20, labelsize=ACL_TICK_LABEL_FONTSIZE)
ax2.tick_params(axis="y", rotation=0, labelsize=ACL_TICK_LABEL_FONTSIZE)
ax2.set_xlabel(ax2.get_xlabel(), fontsize=ACL_AXIS_LABEL_FONTSIZE)
ax2.set_ylabel(ax2.get_ylabel(), fontsize=ACL_AXIS_LABEL_FONTSIZE)
cbar = ax2.collections[0].colorbar
cbar.ax.tick_params(labelsize=ACL_TICK_LABEL_FONTSIZE)
cbar.set_label("Count", fontsize=ACL_AXIS_LABEL_FONTSIZE)

fig2.tight_layout(pad=2.0)
fig2.savefig(FIG2_OUT_PATH, format="pdf", bbox_inches="tight")
plt.show()
print(f"Figure 2 saved to {FIG2_OUT_PATH}")

# ── Figure 3: Per-level Unclear rate by annotator ─────────────────────────────
fig3, ax3 = plt.subplots(figsize=(7, 5.5))

x = np.arange(len(LEVELS))
width = 0.35
unc_a1 = [unclear_by_level[lvl]["a1"] for lvl in LEVELS]
unc_a2 = [unclear_by_level[lvl]["a2"] for lvl in LEVELS]

ax3.bar(x - width/2, unc_a1, width, label="Annotator 1", color=CB_COLORS[4], zorder=2)
ax3.bar(x + width/2, unc_a2, width, label="Annotator 2", color=CB_COLORS[5], zorder=2)
ax3.set_xticks(x)
ax3.set_xticklabels(LEVELS)
ax3.set_ylabel("Proportion Unclear", fontsize=ACL_AXIS_LABEL_FONTSIZE)
ax3.set_title("")
ax3.set_ylim(0, max(max(unc_a1), max(unc_a2)) * 1.4 + 0.01)
ax3.legend(fontsize=ACL_LEGEND_FONTSIZE, frameon=False)
for xi, (u1, u2) in zip(x, zip(unc_a1, unc_a2)):
    ax3.text(xi - width/2, u1 + 0.005, f"{u1:.2f}", ha="center", va="bottom", fontsize=ACL_ANNOTATION_FONTSIZE)
    ax3.text(xi + width/2, u2 + 0.005, f"{u2:.2f}", ha="center", va="bottom", fontsize=ACL_ANNOTATION_FONTSIZE)

fig3.tight_layout(pad=2.0)
fig3.savefig(FIG3_OUT_PATH, format="pdf", bbox_inches="tight")
plt.show()
print(f"Figure 3 saved to {FIG3_OUT_PATH}")

# ── Figure 4: Resolved judgment vs benchmark label (stacked bar) ─────────────
# constrained_layout + fig-level "outside lower center" legend: matches the
# legend-below-plot convention used elsewhere in this repo
# (audit_pipeline/notebooks/figures_consolidated.ipynb), so the legend no
# longer sits on top of the stacked bars.
fig4, ax4 = plt.subplots(figsize=(7, 5.5), constrained_layout=True)

# For each level, tally the four agreement categories among included rows
# Plus: count disagreements and both-unclear as separate slices for completeness
stack_labels = [
    "Ann & Bench: Hateful",       # TP
    "Ann & Bench: Not Hateful",   # TN
    "Ann Hateful / Bench Non-hat (bench FP)",  # FP from benchmark perspective
    "Ann Non-hat / Bench Hateful (bench FN)",  # FN from benchmark perspective
]
stack_colors = [CB_COLORS[5], CB_COLORS[2], CB_COLORS[0], CB_COLORS[1]]

bar_data = {label: [] for label in stack_labels}
level_totals = []

for lvl in LEVELS:
    sub = df_comp[df_comp["coding_level"] == lvl]
    n   = len(sub)
    level_totals.append(n)
    tp_l = ((sub["resolved_binary"] == 1) & (sub["binary_hate"] == 1)).sum()
    fp_l = ((sub["resolved_binary"] == 1) & (sub["binary_hate"] == 0)).sum()
    fn_l = ((sub["resolved_binary"] == 0) & (sub["binary_hate"] == 1)).sum()
    tn_l = ((sub["resolved_binary"] == 0) & (sub["binary_hate"] == 0)).sum()
    bar_data["Ann & Bench: Hateful"].append(tp_l / n)
    bar_data["Ann & Bench: Not Hateful"].append(tn_l / n)
    bar_data["Ann Hateful / Bench Non-hat (bench FP)"].append(fp_l / n)
    bar_data["Ann Non-hat / Bench Hateful (bench FN)"].append(fn_l / n)

bottoms = np.zeros(len(LEVELS))
for label, color in zip(stack_labels, stack_colors):
    vals = np.array(bar_data[label])
    ax4.bar(LEVELS, vals, bottom=bottoms, label=label, color=color, width=0.5)
    bottoms += vals

ax4.set_ylabel("Proportion of Included Items", fontsize=ACL_AXIS_LABEL_FONTSIZE)
ax4.set_title("")
ax4.set_ylim(0, 1.0)
for i, (lvl, n_tot) in enumerate(zip(LEVELS, level_totals)):
    ax4.text(i, 1.02, f"N={n_tot}", ha="center", fontsize=ACL_ANNOTATION_FONTSIZE)

# Legend moved below the plot (was ax-level, upper-right, overlapping the top
# of the stacked bars) -- fig-level legend, two columns of two colors, placed
# outside/below the axes. No tight_layout call: constrained_layout (set above)
# already reserves space for it; calling both together can fight each other.
handles, labels = ax4.get_legend_handles_labels()
fig4.legend(
    handles, labels,
    loc="outside lower center", ncol=2,
    fontsize=ACL_LEGEND_FONTSIZE, frameon=False,
)
fig4.savefig(FIG4_OUT_PATH, format="pdf", bbox_inches="tight")
plt.show()
print(f"Figure 4 saved to {FIG4_OUT_PATH}")


## Cell 7: Reportable Metrics Summary

Consolidated summary of all paper-ready numbers, formatted for direct copy-paste into
the manuscript. Also saved to a plain text file for archival.

In [ ]:
def build_metrics_summary(fpr_df):
    """
    Build the reportable metrics summary as a formatted string.

    Pulls from module-level variables computed in Cells 3–5. All numbers
    are rounded to the precision appropriate for a paper: κ to 3 decimal
    places, proportions to 3 decimal places, percentages to 1 decimal.

    Args:
        fpr_df: DataFrame with columns [level, n_included, fpr, ci_lo, ci_hi]

    Returns:
        str: multi-line formatted summary
    """
    fpr_idx = fpr_df.set_index("level")

    # Per-level kappa formatted
    kappa_line = "   ".join(
        f"{lvl}: {level_kappas[lvl]:.3f}" if not np.isnan(level_kappas[lvl]) else f"{lvl}: N/A"
        for lvl in LEVELS
    )

    # FPR lines
    fpr_lines = []
    for lvl in LEVELS:
        row = fpr_idx.loc[lvl]
        unstable = " [UNSTABLE]" if row["n_included"] < MIN_AGREED_ITEMS else ""
        fpr_lines.append(
            f"    {lvl}: {row['fpr']:.3f} [{row['ci_lo']:.3f}, {row['ci_hi']:.3f}]"
            f"  (N={int(row['n_included'])}){unstable}"
        )

    # Unclear rates (both annotators combined, per level = max of A1 and A2)
    unclear_line = "   ".join(
        f"{lvl}: {max(unclear_by_level[lvl]['a1'], unclear_by_level[lvl]['a2'])*100:.1f}%"
        for lvl in LEVELS
    )

    # Annotator vs benchmark agreement
    bench_level_str = "   ".join(
        f"{lvl}: {level_agree[lvl]*100:.1f}%"
        for lvl in LEVELS
    )

    # Monotonic check
    unstable_levels = [
        f"{lvl} (N={int(fpr_idx.loc[lvl, 'n_included'])})"
        for lvl in LEVELS
        if fpr_idx.loc[lvl, "n_included"] < MIN_AGREED_ITEMS
    ]
    unstable_note = ", ".join(unstable_levels) if unstable_levels else "NONE"

    lines = [
        "╔══════════════════════════════════════════════════════════╗",
        "║  REPORTABLE METRICS FOR PAPER                           ║",
        "╠══════════════════════════════════════════════════════════╣",
        "║  IAA (FPR annotation task)                              ║",
        f"║    Cohen's κ (3-way):              {kappa_3way:.3f}{'':24}║",
        f"║    Cohen's κ (binary, excl. Unclear): {kappa_binary:.3f} (N={n_definitive}){'':12}║",
        f"║    Percent agreement:              {pct_agree*100:.1f}%{'':23}║",
        "║                                                         ║",
        "║  Per-level κ                                            ║",
        f"║    {kappa_line}{'':33}║",
        "║                                                         ║",
        "║  FPR estimates (Wilson 95% CI)                          ║",
    ]
    for fl in fpr_lines:
        lines.append(f"║{fl:<57}║")
    lines += [
        "║                                                         ║",
        "║  Unclear rates (max of two annotators)                  ║",
        f"║    {unclear_line}{'':33}║",
        "║                                                         ║",
        "║  Annotator vs benchmark agreement                       ║",
        f"║    Overall: {overall_agree_rate*100:.1f}%{'':45}║",
        f"║    {bench_level_str}{'':33}║",
        "║                                                         ║",
        f"║  Monotonic FPR? [{mono_label:<41}]║",
        f"║  Unstable cells (N < {MIN_AGREED_ITEMS}): {unstable_note:<34}║",
        "╚══════════════════════════════════════════════════════════╝",
    ]
    return "\n".join(lines)


summary_text = build_metrics_summary(fpr_df)
print(summary_text)

# Save to file
METRICS_OUT_PATH.write_text(summary_text + "\n")
print(f"\nMetrics saved to {METRICS_OUT_PATH}")